# Train the legal rhetorical-role labeller on Google Colab

Connect to a **hosted CPU runtime** and choose **Runtime → Run all**. This notebook trains a seven-role hierarchical BiLSTM-CRF from random initialization. It uses the 50 annotated judgments in the public repository. The validation fit and final fit each learn their own vocabulary and parameters.

The final cell saves a ZIP in Colab's Files panel. Download it before the runtime disconnects.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import platform
import subprocess
import sys
import tempfile
import time
from zipfile import ZIP_DEFLATED, ZipFile

import google.colab
import sklearn
import torch
from sklearn.metrics import classification_report

content = Path("/content")
if not content.is_dir():
    raise RuntimeError("Connect this notebook to a hosted Google Colab runtime.")
work = Path(tempfile.mkdtemp(prefix="legal-lexrank-", dir=content))
source = work / "source"
artifacts = work / "artifacts"
artifacts.mkdir()
seed = 42
epochs = 8
started = datetime.now(timezone.utc)
torch.set_num_threads(min(4, torch.get_num_threads()))
torch.use_deterministic_algorithms(True)

# This public revision contains the source and corpus for a fresh training run.
repository = "https://github.com/SrikarKorrapati/Extractive-Summarisation-of-Legal-Document.git"
revision = "27fe06544ba88b56767c324e23507a0f89fc50d3"
subprocess.run(["git", "init", "--quiet", str(source)], check=True)
subprocess.run(["git", "fetch", "--quiet", "--depth=1", repository, revision], cwd=source, check=True)
subprocess.run(["git", "checkout", "--quiet", "--detach", "FETCH_HEAD"], cwd=source, check=True)
actual_revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=source, text=True).strip()
assert actual_revision == revision
sys.path.insert(0, str(source))

from legal_lexrank.data import ROLES, load_cases, split_case_ids
from legal_lexrank.labeller import fit_labeller, load_labeller

print(f"PyTorch {torch.__version__} · training device CPU · source {revision[:12]}")

## Split by judgment

Set aside ten judgments for testing. Within the remaining 40, use 32 for the initial fit and eight to measure validation performance. The final fit uses all 40 training judgments. Gold labels are attached to annotated sentences, whose boundaries can differ from those used by the summarizer.

In [ ]:
cases = load_cases(source)
split = split_case_ids(cases, seed=seed)
counts = {name: len(ids) for name, ids in split.items()}
assert counts == {"fit": 32, "validation": 8, "train": 40, "test": 10}
assert set(split["fit"]).isdisjoint(split["validation"])
assert set(split["fit"]) | set(split["validation"]) == set(split["train"])
assert set(split["train"]).isdisjoint(split["test"])
assert set(split["train"]) | set(split["test"]) == set(cases)
print(counts)

def labelled_documents(ids):
    documents = []
    for case_id in ids:
        case = cases[case_id]
        annotations = [(sentence, role) for sentence, role in
                       zip(case.labelled_sentences, case.labels, strict=True) if role is not None]
        if not annotations:
            raise ValueError(f"Missing annotations for {case_id}")
        sentences, roles = zip(*annotations)
        documents.append((sentences, roles))
    return documents

def evaluate_roles(model, ids):
    expected, predicted = [], []
    for case_id in ids:
        case = cases[case_id]
        predictions = model.predict(case.labelled_sentences)
        for gold, guess in zip(case.labels, predictions, strict=True):
            if gold is not None:
                expected.append(gold)
                predicted.append(guess)
    return classification_report(expected, predicted, labels=ROLES, output_dict=True, zero_division=0)

## Train and evaluate

The word-level BiLSTM encodes each sentence. A second BiLSTM processes sentence order within a judgment. The CRF learns transitions between roles and uses Viterbi decoding to predict the full role sequence. Both fits use eight epochs on CPU.

In [ ]:
reports = {}
durations = {}
for phase, training, held_out in (("validation", "fit", "validation"), ("test", "train", "test")):
    final_fit = phase == "test"
    checkpoint = artifacts / "role_labeller.pt" if final_fit else work / "validation.pt"
    tick = time.monotonic()
    print(f"Starting {phase} fit on {len(split[training])} cases", flush=True)
    fitted = fit_labeller(labelled_documents(split[training]), ROLES, seed, checkpoint, epochs=epochs)
    durations[phase] = round(time.monotonic() - tick, 3)
    # Evaluate the just-written checkpoint, including serialisation and reload.
    model = load_labeller(checkpoint)
    reports[phase] = evaluate_roles(model, split[held_out])
    print(f"{phase}: accuracy {reports[phase]['accuracy']:.4f}, "
          f"macro F1 {reports[phase]['macro avg']['f1-score']:.4f}, "
          f"fit time {durations[phase]:.1f}s", flush=True)

## Download the trained model

After the next cell completes, open **Files** in the Colab sidebar and download **legal_lexrank_role_labeller.zip**. It contains the final 40-case checkpoint, case split, per-role validation and test metrics, and a runtime record with the SHA-256 of the checkpoint.

In [ ]:
def save_json(name, obj):
    (artifacts / name).write_text(json.dumps(obj, indent=2, allow_nan=False) + "\n", encoding="utf-8")

final_checkpoint = artifacts / "role_labeller.pt"
save_json("protocol.json", {"source_url": repository, "source_revision": revision, "case_ids": split,
                            "roles": list(ROLES), "seed": seed, "epochs_per_fit": epochs})
save_json("role_metrics.json", reports)
save_json("runtime.json", {
    "platform": "Google Colab", "device": str(next(model.parameters()).device),
    "colab_release_tag": os.environ.get("COLAB_RELEASE_TAG"),
    "os": platform.platform(), "python": platform.python_version(),
    "torch": str(torch.__version__), "sklearn": sklearn.__version__, "torch_threads": torch.get_num_threads(),
    "started_utc": started.isoformat(), "finished_utc": datetime.now(timezone.utc).isoformat(),
    "fit_seconds": durations, "checkpoint_sha256": hashlib.sha256(final_checkpoint.read_bytes()).hexdigest(),
})
archive = content / "legal_lexrank_role_labeller.zip"
staging = work / archive.name
with ZipFile(staging, "w", compression=ZIP_DEFLATED) as file:
    for artifact in sorted(artifacts.iterdir()):
        file.write(artifact, artifact.name)
staging.replace(archive)
print(f"Training complete. Download {archive}")